# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [41]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.15 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

**TODO:** Replace this text with an evidence map: one bullet for each of Lectures 01 to 06. For 01, 02, 04, 05, and 06, name one concrete technique or file and say in one sentence where this notebook or its outputs use it. For 03, complete the NumPy calculation after Task 4.3 and explain your axis choice in one sentence here. The lectures cover the command line and Python basics (01); Git, functions, and files (02); environments, shell pipelines, and NumPy (03); Jupyter and labeled pandas data (04); data contracts, cleaning, and validation (05); and joins, reshaping, and concatenation (06). Do not paste earlier solutions.

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [42]:
%%bash
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [43]:
#write a function, such as profile(name, table), that returns the four values in README
def table_profile(name, table):
    """Returns a dictionary with table: file name, rows: data rows without header, columns, and exact_duplicate_rows: rows identical to earlier row."""
    return dict(
            table = name,
            rows = len(table),
            columns = len(table.columns),
            exact_duplicate_rows = table.duplicated().sum()
        )
# Task 1.2, call it once per name in manifest["files"] with raw[Path(name).stem], and save the results to PROFILE_PATH.
results = [table_profile(name, raw[Path(name).stem]) for name in manifest["files"]]
pd.DataFrame(results).to_csv(PROFILE_PATH, index=False)

## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

**TODO:** Replace this text with the contract: the intended grain and key (single or composite) of each cleaned input table, and the cardinality you expect for each Task 4.4 join. Describe visits after stacking and keeping the later recheck, sites after keeping current records, and the long-form A1c join keys. Note raw exact repeats, rechecks, and site versions that violate the intended cleaned keys (README Task 2.1).

### 2.2 Stack the two visit files

In [44]:
#rename wt_kg, add source_file to each visit table, and stack them, north first (README Task 2.2).
visits_south_copy = raw["visits_south"].rename(columns={"wt_kg": "weight_kg"}, inplace=False)
visits_south_copy["source_file"] = "visits_south.csv"
visits_north_copy = raw["visits_north"].copy(deep=True)
visits_north_copy["source_file"] = "visits_north.csv"
visits_raw = pd.concat([visits_north_copy, visits_south_copy], ignore_index = True)

### 2.3 Audit the raw tables

In [45]:
#count the 10 issues in README Task 2.3, in order, as a DataFrame with the columns issue and count, and save it to AUDIT_PATH.

##Gender values not a code:
allowed_gender_codes = ['M', 'F', 'X']
gender_not_a_code = ((raw["patients"]["gender"].str.strip().notnull()) & (raw["patients"]["gender"].str.strip() != "") &(~raw["patients"]["gender"].isin(allowed_gender_codes))).sum()

##Empty gender values after str.strip()
empty_gender = ((raw["patients"]["gender"].str.strip().isnull()) | (raw["patients"]["gender"].str.strip() == "")).sum()

###Birth year sentinel 9999
birth_year_sentinel = (pd.to_numeric(raw["patients"]["birth_year"], errors="coerce") == 9999).sum()

##Site codes needing normalization
normalized_site_codes = raw["patients"]["site_code"].str.upper().str.replace("-","").str.replace(" ","")
site_codes_needing_normalization = normalized_site_codes != raw["patients"]["site_code"]

##Rows with a repeated site_code in sites
repeated_site_codes = raw["sites"].duplicated(subset = ["site_code"], keep = False).sum()

##Visit labels that are not baseline or week12
invalid_visit_labels = (~visits_raw["visit"].isin(["baseline", "week12"])).sum()

##SBP sentinel 999
sentinel_sbp = (pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce") == 999).sum()

##SBP outside 70 through 250
sbp_outside_range = ((pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce") != 999) & ((pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce") < 70) | (pd.to_numeric(visits_raw["sbp_mmhg"], errors="coerce") > 250))).sum()

##Date parse failures
not_exact_date = ((~visits_raw["visit_date"].str.fullmatch(r'[0-9]{4}-[0-9]{2}-[0-9]{2}', na = False)) | (pd.to_datetime(visits_raw["visit_date"], format  = "%Y-%m-%d", errors = "coerce").isna())).sum()

##Rows in repeated patient_id and visit pairs
visit_dict = {"bl":"baseline", "baseline":"baseline", "week 12": "week12", "week12": "week12", "wk12": "week12"}
normalized_visits = visits_raw["visit"].str.lower().str.strip().map(visit_dict)
duplicate_test = pd.DataFrame({
    "patient_id": visits_raw["patient_id"],
    "visit": normalized_visits
})
repeated_patient_visit_pairs = (duplicate_test.duplicated(subset=['patient_id', 'visit'], keep=False)).sum()

issue_audit = pd.DataFrame({
    "issue": [
        "patients: gender values that are not a code",
        "patients: empty gender",
        "patients: birth_year sentinel 9999",
        "patients: site codes needing normalization",
        "sites: rows with a repeated site_code",
        "visits: visit labels that are not baseline or week12",
        "visits: sbp sentinel 999",
        "visits: sbp outside 70 through 250",
        "visits: date parse failures",
        "visits: rows in repeated patient_id and visit pairs"
    ],
    "count": [
        gender_not_a_code,
        empty_gender,
        birth_year_sentinel,
        site_codes_needing_normalization.sum(),
        repeated_site_codes,
        invalid_visit_labels,
        sentinel_sbp,
        sbp_outside_range,
        not_exact_date,
        repeated_patient_visit_pairs
    ]
})

assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."

issue_audit.to_csv(AUDIT_PATH, index=False)

## Task 3: Clean

### 3.1 Clean the patients

In [46]:
#list the gender spellings with value_counts(), then build patients_clean from
# raw["patients"].copy(deep=True) with steps 1 to 5 in README Task 3.1.
patients_clean = raw["patients"].copy(deep=True)
patients_clean = patients_clean.drop_duplicates()
patients_clean["gender"].value_counts()
patients_clean["gender"] = patients_clean["gender"].str.strip().str.lower()
gender_dict = {"female": "F", "f": "F", "woman": "F", "male": "M", "man": "M", "m": "M", "x": "X", "unknown": pd.NA, "": pd.NA, "nb": "X", "nonbinary":"X", "non-binary":"X"}
patients_clean["gender"] = patients_clean["gender"].map(gender_dict)
patients_clean["gender"] = patients_clean["gender"].astype("category")

patients_clean["birth_year"] = (pd.to_numeric(patients_clean["birth_year"], errors = "coerce")).replace(9999, pd.NA).astype("Int64")

patients_clean["site_code"] = patients_clean["site_code"].str.upper().str.replace("-","").str.replace(" ","")

patients_clean.reset_index(drop=True, inplace=True)

summary = pd.DataFrame({
    "metric": ["patients", "birth_year missing", "birth_year min", "birth_year max", "site_code values", "patients at N01", "patients at S02", "patients at E04"],
    "value": [
        len(patients_clean), 
        patients_clean["birth_year"].isna().sum(),
        patients_clean["birth_year"].min(), 
        patients_clean["birth_year"].max(), 
        patients_clean["site_code"].nunique(), 
        (patients_clean["site_code"] == "N01").sum(), 
        (patients_clean["site_code"] == "S02").sum(), 
        (patients_clean["site_code"] == "E04").sum()
    ]
})

summary.to_csv(PATIENTS_SUMMARY_PATH, index=False)

In [47]:
#show patients_clean["gender"].cat.categories, save the patients summary to
# PATIENTS_SUMMARY_PATH, and save the gender value_counts(dropna=False) with a code column from
# .cat.codes to GENDER_CATEGORIES_PATH, as in README Task 3.1.
gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)

### 3.2 Clean the visits

In [48]:
#build visits_clean from visits_raw.copy(deep=True) with steps 1 to 5 in README Task 3.2.
visits_clean = visits_raw.copy(deep=True)
visits_clean = visits_clean.drop_duplicates(keep = 'first')
visits_clean["visit"].value_counts()
visits_clean["visit"] = visits_clean["visit"].str.strip().str.lower().map(visit_dict)
visits_clean["sbp_mmhg"] = pd.to_numeric(visits_clean["sbp_mmhg"], errors="coerce").astype("Int64")
visits_clean["sbp_mmhg"] = visits_clean["sbp_mmhg"].where((visits_clean["sbp_mmhg"].between(70,250)))
visits_clean["weight_kg"] = pd.to_numeric(visits_clean["weight_kg"], errors="coerce")
exact_date = visits_clean["visit_date"].str.fullmatch(r'[0-9]{4}-[0-9]{2}-[0-9]{2}')
visits_clean["visit_date"] = pd.to_datetime(visits_clean["visit_date"].where(exact_date), format='%Y-%m-%d', errors='coerce')

In [49]:
#step 6, show the repeated patient_id and visit pairs, then keep the later reading.
repeat_pairs = visits_clean.duplicated(subset = ["patient_id", "visit"], keep=False)
print(visits_clean[repeat_pairs].sort_values("visit_time"))
visits_clean = visits_clean.sort_values("visit_time")
visits_clean = visits_clean.drop_duplicates(subset=["patient_id", "visit"], keep="last")

      patient_id     visit visit_date visit_time  sbp_mmhg  weight_kg  \
41284     P14559  baseline 2026-01-23      08:10       143       89.3   
11999     P13588    week12 2026-04-23      08:15       143       78.9   
42550     P15803  baseline 2026-02-19      08:15       147      112.5   
41283     P14559  baseline 2026-01-23      08:20       142       89.3   
10787     P12183  baseline 2026-01-26      08:30       122       70.7   
...          ...       ...        ...        ...       ...        ...   
34917     P08493  baseline 2026-01-13      16:25       157       66.0   
6011      P06732    week12 2026-05-21      16:35       115       88.9   
16689     P18973  baseline 2026-02-09      16:35       165      108.6   
49932     P22899    week12 2026-03-26      16:35       160      103.6   
12876     P14599  baseline 2026-01-07      16:40       154      102.4   

            source_file  
41284  visits_south.csv  
11999  visits_north.csv  
42550  visits_south.csv  
41283  visits_south

In [50]:
#step 7, sort, renumber, keep the six columns, then save the visits summary to
# VISITS_SUMMARY_PATH (README Task 3.2).
visits_clean = visits_clean.sort_values(["patient_id", "visit"]).reset_index(drop=True)
visits_clean = visits_clean[
    ["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "source_file"]
]
visits_summary = pd.DataFrame({
    "metric": [
        "visits",
        "baseline visits",
        "week12 visits",
        "visit_date missing",
        "visit_date min",
        "visit_date max",
        "sbp_mmhg missing",
        "sbp_mmhg min",
        "sbp_mmhg max",
        "weight_kg missing",
        "weight_kg mean",
        "visits from visits_north.csv",
        "visits from visits_south.csv"
    ],
    "value": [
        len(visits_clean),
        (visits_clean["visit"] == "baseline").sum(),
        (visits_clean["visit"] == "week12").sum(),
        (visits_clean["visit_date"].isna()).sum(),
        visits_clean["visit_date"].min(),
        visits_clean["visit_date"].max(),
        (visits_clean["sbp_mmhg"].isna()).sum(),
        visits_clean["sbp_mmhg"].min(),
        visits_clean["sbp_mmhg"].max(),
        (visits_clean["weight_kg"].isna()).sum(),
        round(visits_clean["weight_kg"].mean(), 2),
        (visits_clean["source_file"] == "visits_north.csv").sum(),
        (visits_clean["source_file"] == "visits_south.csv").sum()
    ]
})

visits_summary.to_csv(VISITS_SUMMARY_PATH, index = False)

### 3.3 Record the decisions

In [51]:
#one row per issue in README Task 3.3, with table and issue copied, and your own
# decision and one-sentence reason, then save it to DECISION_LOG_PATH.
decision_table = pd.DataFrame({
                                "table": [
                                  "patients",
                                  "patients",
                                  "patients",
                                  "patients",
                                  "visits",
                                  "visits",
                                  "visits",
                                  "visits",
                                  "sites"
                              ], 
                                "issue": [
                                   "exact duplicate rows",
                                   "gender spellings that are not a code",
                                   "site codes needing normalization",
                                   "birth_year sentinel 9999",
                                   "weight column named wt_kg in visits_south.csv",
                                   "sbp sentinel 999 and readings outside 70 through 250",
                                   "impossible and loosely written visit dates",
                                   "repeated patient_id and visit pairs",
                                   "repeated site_code rows"
                              ], 
                                "decision": [
                                   "Removed duplicate rows, keeping the first",
                                   "Reviewed values, recoded to M, F, X, or missing",
                                   "Cleaned site_code so all of same format (upper case, no hyphens)",
                                   "Converted sentinel 9999 to NA",
                                   "Changed name to weight_kg to match the visits_north.csv file",
                                   "Changed sentinel 999 and readings outside 70-250 to NA",
                                   "Changed impossible and loosely written dates to missing",
                                   "Sorted repeated patient ID and visit pairs by visit_time and kept the later entry",
                                   "Keep only the current sites"


                               ],
                                "reason": [
                                   "Each patient only needs to be listed once- this table contains no temporal data",
                                   "Allows easier data review and analysis",
                                   "Allows for easier data review and comparison/analysis",
                                   "Changes coding of birth_year sentinel to a missing value understood by pandas",
                                   "Matching visits_north.csv allows concatenation of the two files",
                                   "Changes unknown values and implausible values to missing",
                                   "Only keeps date values that we are confident we know because they are coded correctly",
                                   "Duplicate patient ID and visit pairs indicates repeat reading- kept most recent reading",
                                   "Current and retired sites share site codes- keeping current only so that there aren't duplicate matches when joining datasets"
                               ]
})
decision_table.to_csv(DECISION_LOG_PATH, index = False)

## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [52]:
# TODO: melt raw["a1c_wide"], map the visit labels, drop the rows with no result,
# then save the per-visit summary to A1C_SUMMARY_PATH (README Task 4.1).
a1c_long = None

### 4.2 Check the keys before joining

In [53]:
# TODO: steps 1 and 2, catch the MergeError, show the repeated site rows, and build sites_current.
sites = raw["sites"]
sites_current = None

In [54]:
# TODO: step 3, outer merge visits_clean and a1c_long with indicator=True and show the _merge counts,
# then save the seven counts in README Task 4.2 to MERGE_AUDIT_PATH.

### 4.3 Pivot blood pressure to one row per patient

In [55]:
# TODO: pivot visits_clean, reset the index, clear columns.name, add sbp_change, then save the
# SAMPLE_IDS rows to SBP_WIDE_SAMPLE_PATH (README Task 4.3).
sbp_wide = None

#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [56]:
# TODO: select complete pairs, convert to a NumPy array, and display its shape and two per-visit means.

### 4.4 Join the analysis table

In [57]:
# TODO: three left merges onto visits_clean, each with validate=, then the columns and dtypes
# in README Task 4.4. A MergeError means an earlier step left a problem: see HINTS.md.
analysis = None

## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [58]:
# TODO: collect named True/False checks in a Series (README Task 5.1), display it, and stop with
# assert checks.all(), checks[~checks] before the analysis extract is saved.
# A failed check means an earlier step left a problem: print checks[~checks] and see HINTS.md.

### 5.2 Save the analysis extract

In [59]:
# TODO: keep the analysis rows whose patient_id is in SAMPLE_IDS, renumber them, and save them to
# ANALYSIS_SAMPLE_PATH with index=False (README Task 5.2).

### 5.3 Read the file back

In [60]:
# TODO: read ANALYSIS_SAMPLE_PATH back with its dtypes and parse_dates=["visit_date"], and print
# whether it equals analysis_sample.